# Bronze Layer - RetailMax Data Pipeline

Este notebook implementa la capa Bronze del pipeline de datos con arquitectura Medallion.

**Objetivo:**
- Extraer datos de PostgreSQL
- Agregar columnas de auditoría
- Convertir a formato Parquet
- Cargar al contenedor bronze de Azure Data Lake Storage Gen2

## Instalación de librerias

In [113]:
%pip install pandas sqlalchemy psycopg2-binary pyarrow azure-storage-file-datalake
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [114]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2
import pyarrow as pa
import pyarrow.parquet as pq
import uuid
from datetime import datetime
from azure.storage.filedatalake import DataLakeServiceClient
from pathlib import Path
import os
from dotenv import load_dotenv

## 2. Configuración

In [115]:
# Configuración de PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    user="postgres",
    password="postgres",
    database="retailmax"
)
# Configuración de Azure Storage
AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY = os.getenv("AZURE_STORAGE_KEY")
AZURE_FILE_SYSTEM = "bronze"

## 3. Definición de tablas a procesar

In [116]:
# Lista de tablas a procesar
tables = [
    "mstr_articulos",
    "mstr_proveedores",
    "mstr_tiendas",
    "crm_miembros",
    "fact_ventas",
    "inv_stock_diario",
    "fact_devoluciones"
]

## 4. Funciones auxiliares

In [117]:
def get_azure_datalake_client():

    service_client = DataLakeServiceClient(
        account_url=f"https://{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential=AZURE_STORAGE_KEY
    )
    return service_client

def add_audit_columns(df, batch_id):
    df_copy = df.copy()
    df_copy['batch_id'] = batch_id
    df_copy['carga_timestamp'] = datetime.now()
    df_copy['carga_fecha'] = datetime.now().date()
    return df_copy

def upload_to_adls(service_client, file_system_name, file_path, directory_name):

    file_system_client = service_client.get_file_system_client(file_system_name)
    directory_client = file_system_client.get_directory_client(directory_name)
    
    # Crear directorio si no existe
    try:
        directory_client.create_directory()
    except:
        pass
    
    # Subir archivo
    file_name = os.path.basename(file_path)
    file_client = directory_client.get_file_client(file_name)
    
    with open(file_path, "rb") as data:
        file_client.upload_data(data, overwrite=True)

def process_table(conn, table_name, batch_id, service_client, file_system_name, temp_dir):
    print(f"Leyendo {table_name.upper()}...")
    
    # Leer tabla completa de PostgreSQL usando psycopg2
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql(query, conn)
    
    # Agregar columnas de auditoría
    df = add_audit_columns(df, batch_id)
    
    print("Exportando a Parquet...")
    
    # Crear directorio temporal si no existe
    Path(temp_dir).mkdir(parents=True, exist_ok=True)
    
    # Exportar a Parquet usando fastparquet para evitar el conflicto de PyArrow
    parquet_file = os.path.join(temp_dir, f"{table_name}.parquet")
    try:
        # Intentar con fastparquet primero
        df.to_parquet(parquet_file, index=False, engine="fastparquet")
    except ImportError:
        # Si fastparquet no está instalado, usar pyarrow con manejo de errores
        try:
            # Limpiar el registro de extensiones de PyArrow
            import pyarrow as pa
            if hasattr(pa, "unregister_extension_type"):
                try:
                    pa.unregister_extension_type("pandas.period")
                except:
                    pass
            df.to_parquet(parquet_file, index=False, engine="pyarrow")
        except Exception as e:
            # Si falla, intentar sin especificar engine
            df.to_parquet(parquet_file, index=False)
    
    print("Subiendo a ADLS...")
    
    # Subir a ADLS
    upload_to_adls(service_client, file_system_name, parquet_file, table_name.upper())
    
    # Eliminar archivo temporal
    os.remove(parquet_file)
    
    print(f"Tabla {table_name.upper()} cargada correctamente.")
    print("-" * 33)

## 5. Ejecución del proceso Bronze

In [118]:
# Directorio temporal para archivos Parquet
TEMP_DIR = "temp_parquet"

# Generar batch_id único para esta ejecución
batch_id = str(uuid.uuid4())

print(f"Batch ID: {batch_id}")
print("=" * 50)

# Contador de tablas procesadas exitosamente
tables_processed = 0

Batch ID: a3cde604-21f6-40cc-8c1b-6d71b2bf0d8c


## 6. Conexión a PostgreSQL

In [119]:
try:
    print("Conectando a PostgreSQL...")
    # La conexión ya está establecida en la celda de configuración (cell 5)
    print("Conexión a PostgreSQL establecida exitosamente.")
    print("-" * 33)
except Exception as e:
    print(f"Error al conectar a PostgreSQL: {e}")
    raise

Conectando a PostgreSQL...
Conexión a PostgreSQL establecida exitosamente.
---------------------------------


## 7. Conexión a Azure Data Lake Storage

In [120]:
try:
    print("Conectando a Azure Data Lake Storage...")
    service_client = get_azure_datalake_client()
    print("Conexión a ADLS establecida exitosamente.")
    print("-" * 33)
except Exception as e:
    print(f"Error al conectar a ADLS: {e}")
    raise

Conectando a Azure Data Lake Storage...
Conexión a ADLS establecida exitosamente.
---------------------------------


## 8. Procesamiento de tablas

In [121]:
try:
    for table in tables:
        try:
            process_table(
                conn=conn,
                table_name=table,
                batch_id=batch_id,
                service_client=service_client,
                file_system_name=AZURE_FILE_SYSTEM,
                temp_dir=TEMP_DIR
            )
            tables_processed += 1
        except Exception as e:
            print(f"Error al procesar tabla {table.upper()}: {e}")
            continue
            
except Exception as e:
    print(f"Error general en el procesamiento: {e}")
    raise

Leyendo MSTR_ARTICULOS...
Exportando a Parquet...


C:\Users\PradoV09\AppData\Local\Temp\ipykernel_9824\2098524618.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Subiendo a ADLS...
Error al procesar tabla MSTR_ARTICULOS: Server failed to authenticate the request. Please refer to the information in the www-authenticate header.
RequestId:7dceb688-401f-0059-2086-1f5174000000
Time:2026-07-29T18:18:16.3119606Z
ErrorCode:NoAuthenticationInformation
Leyendo MSTR_PROVEEDORES...
Exportando a Parquet...
Subiendo a ADLS...


C:\Users\PradoV09\AppData\Local\Temp\ipykernel_9824\2098524618.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Error al procesar tabla MSTR_PROVEEDORES: Server failed to authenticate the request. Please refer to the information in the www-authenticate header.
RequestId:7dceb738-401f-0059-4f86-1f5174000000
Time:2026-07-29T18:18:16.8575676Z
ErrorCode:NoAuthenticationInformation
Leyendo MSTR_TIENDAS...
Exportando a Parquet...
Subiendo a ADLS...


...

Exportando a Parquet...
Subiendo a ADLS...
Error al procesar tabla FACT_VENTAS: Server failed to authenticate the request. Please refer to the information in the www-authenticate header.
RequestId:7dcec82b-401f-0059-7e86-1f5174000000
Time:2026-07-29T18:18:34.4908964Z
ErrorCode:NoAuthenticationInformation
Leyendo INV_STOCK_DIARIO...


C:\Users\PradoV09\AppData\Local\Temp\ipykernel_9824\2098524618.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exportando a Parquet...
Subiendo a ADLS...
Error al procesar tabla INV_STOCK_DIARIO: Server failed to authenticate the request. Please refer to the information in the www-authenticate header.
RequestId:7dceccd3-401f-0059-2086-1f5174000000
Time:2026-07-29T18:18:38.4099202Z
ErrorCode:NoAuthenticationInformation
Leyendo FACT_DEVOLUCIONES...


C:\Users\PradoV09\AppData\Local\Temp\ipykernel_9824\2098524618.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exportando a Parquet...
Subiendo a ADLS...
Error al procesar tabla FACT_DEVOLUCIONES: Server failed to authenticate the request. Please refer to the information in the www-authenticate header.
RequestId:7dcece85-401f-0059-5286-1f5174000000
Time:2026-07-29T18:18:39.4746307Z
ErrorCode:NoAuthenticationInformation


## 9. Limpieza y cierre de conexiones

In [122]:
try:
    # Cerrar conexión a PostgreSQL
    conn.close()
    print("Conexión a PostgreSQL cerrada.")
    
    # Eliminar directorio temporal si existe
    if os.path.exists(TEMP_DIR):
        os.rmdir(TEMP_DIR)
        print("Directorio temporal eliminado.")
        
except Exception as e:
    print(f"Error durante la limpieza: {e}")

Conexión a PostgreSQL cerrada.
Error durante la limpieza: [WinError 145] The directory is not empty: 'temp_parquet'


## 10. Resumen de ejecución

In [123]:
print("=" * 50)
print("Proceso Bronze finalizado exitosamente.")
print(f"Total de tablas procesadas: {tables_processed}")
print(f"Batch ID: {batch_id}")
print("=" * 50)

Proceso Bronze finalizado exitosamente.
Total de tablas procesadas: 0
Batch ID: a3cde604-21f6-40cc-8c1b-6d71b2bf0d8c
